# Unificacion de bases de dinamica forestal (DBH / recensos)

Este cuaderno homogeniza y concatena las bases de **dinamica forestal** (diametro a
la altura del pecho, DBH, por censo) de los 4 sitios de trabajo:

1. **Guacamayos** (`GUACAMAYOS_ForestPlot_2025_jh.xlsx`) — formato ForestPlots, una hoja por parcela
2. **Oyacachi** (`Oyacachi_forestplot_2025_jh_GP.xlsx`) — formato ForestPlots, una hoja por parcela
3. **Galeras** (`Base_Galeras_vK.xlsx`)
4. **Selva Viva** (`Base_SelvaViva_v2.xlsx`)
5. **Sumaco** (`Sumaco_17092025.xlsx`)

Guacamayos y Oyacachi **se unifican primero entre si** (seccion 3) porque vienen en
formato ForestPlots crudo (una hoja por parcela, bloques de censo en horizontal y un
numero variable de censos). Esa base intermedia se guarda aparte
(`Base_unificada_ForestPlots_Guacamayos_Oyacachi.xlsx`) y recien despues se concatena
con las otras tres.

**Reglas aplicadas:**

- Se eliminan las columnas relacionadas con **rasgos/muestras** (leaf/wood sample,
  herbario, voucher, crust thickness, wet/dry wood weight, dendrometro, altura, etc.),
  porque esa informacion ya esta en la base de rasgos trabajada previamente. La
  seccion 12 imprime la lista de columnas descartadas en cada base para poder revisarlo.
- Se conserva: `PlotID`, `Plot`, `Subplot`, IDs viejos (`treeID`, `Old_Tag`),
  `new_tree_ID`, fechas y años de muestreo, **DBH de los 3 censos**, banderas de
  mortalidad y comentarios.
- Se agregan `family`/`genus`/`species` porque son necesarias para unir esta base con
  la de rasgos mas adelante y para interpretar los DBH faltantes (un DBH vacio puede
  deberse a que el arbol ya estaba muerto).
- **Fechas**: se homogenizan todas a `datetime` (seccion 9), con imputacion en 3
  niveles (moda de la parcela → moda del sitio-año → moda de la campaña de la base) y
  una columna `date_censusN_origen` que dice de donde salio cada fecha. Al final se
  imprime el **aviso de las fechas que siguen faltando**.
- **Filtro principal**: se conservan solo los **plots que si tienen recenso** reciente
  (2023, 2024 o 2025 segun el sitio), mas una lista de **excepciones forzadas**
  (`PLOTS_FORZADOS`) que se conservan aunque no tengan recenso porque hacen falta
  para el analisis posterior.


## 1. Importar librerias, definir rutas y parametros

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

# Carpeta donde estan los archivos originales (ajustar si es necesario)
DATA_DIR = Path('.')
OUT_DIR  = Path('.')

# --- Archivos crudos de ForestPlots (una hoja por parcela) ---
FILES_FP = {
    "Guacamayos": DATA_DIR / "GUACAMAYOS_ForestPlot_2025_jh.xlsx",
    "Oyacachi":   DATA_DIR / "Oyacachi_forestplot_2025_jh_GP.xlsx",
}

# --- Bases ya tabuladas (una sola hoja) ---
FILES = {
    "Galeras":   DATA_DIR / "Base_Galeras_vK.xlsx",
    "SelvaViva": DATA_DIR / "Base_SelvaViva_v2.xlsx",
    "Sumaco":    DATA_DIR / "Sumaco_17092025.xlsx",
}

SHEETS = {
    "Galeras":   "Galeras",
    "SelvaViva": "SelvaViva2024",
    "Sumaco":    "final",
}

# Archivo intermedio con Guacamayos + Oyacachi ya unificados
OUT_GCM_OYA = OUT_DIR / "Base_unificada_ForestPlots_Guacamayos_Oyacachi.xlsx"

# ----------------------------- PARAMETROS -----------------------------------
# Año a partir del cual un censo se considera "recenso reciente"
ANIO_RECENSO_MIN = 2023

# Hojas de los archivos ForestPlots que NO son parcelas o que son duplicados.
# 'OYA-81_GP' es una version alternativa (mas ancha) de la misma parcela OYA-81:
# tiene los mismos tags pero le falta el censo 1, por eso se excluye.
HOJAS_EXCLUIDAS_FP = {"headers_treedata", "OYA-81_GP"}

# Parcelas que se conservan SI O SI aunque no tengan recenso reciente,
# porque hacen falta para los analisis posteriores.
PLOTS_FORZADOS = ["GAL_20", "GAL_29", "SUM_13"]

# Al elegir el censo intermedio (censo2) entre varios candidatos se prefiere el mas
# reciente, siempre que tenga al menos este porcentaje de los DBH del mejor candidato.
UMBRAL_INTERMEDIO = 0.8

# Guardar las columnas de control de calidad de fechas (date_censusN_origen)
INCLUIR_COLS_CALIDAD = True

print("Parametros cargados. Recenso reciente = a partir de", ANIO_RECENSO_MIN)
print("Parcelas forzadas (se conservan sin recenso):", ", ".join(PLOTS_FORZADOS))

Parametros cargados. Recenso reciente = a partir de 2023
Parcelas forzadas (se conservan sin recenso): GAL_20, GAL_29, SUM_13


## 2. Esquema homogeneizado (solo dinamica forestal, sin rasgos)

Esquema comun al que se mapean los nombres originales de cada base. Se agregan
`year_censusN` porque algunas campañas de Guacamayos/Oyacachi vienen con el año
solamente (sin dia ni mes) y esa informacion no se debe perder.

In [2]:
CORE_COLUMNS = [
    "dataset_source", "Site", "PlotID", "Plot", "Subplot",
    "treeID", "Old_Tag", "new_tree_ID",
    "family", "genus", "species",
    "date_census1", "year_census1", "dbh_census1", "dead_census1", "comment_census1",
    "date_census2", "year_census2", "dbh_census2", "dead_census2", "comment_census2",
    "date_census3", "year_census3", "dbh_census3", "dead_census3", "comment_census3",
    "tree_notes", "censos_intermedios", "comment_taxo",
    "source_file",
]

COLS_DATE = ["date_census1", "date_census2", "date_census3"]
COLS_YEAR = ["year_census1", "year_census2", "year_census3"]
COLS_DBH  = ["dbh_census1", "dbh_census2", "dbh_census3"]


def empty_core_df(n):
    """DataFrame vacio (NaN) con el esquema homogeneizado y n filas."""
    return pd.DataFrame({c: [np.nan] * n for c in CORE_COLUMNS})


def concat_notna(*series_list):
    """Concatena varias columnas de texto en una sola, ignorando los valores vacios.
    Util para juntar comentarios repartidos en varias columnas."""
    df_tmp = pd.concat(series_list, axis=1)
    return df_tmp.apply(
        lambda row: " | ".join([str(v) for v in row if pd.notna(v) and str(v).strip() != ""]),
        axis=1
    ).replace("", np.nan)


def columnas_descartadas(df_original, usadas):
    """Devuelve las columnas del archivo original que NO se llevaron al esquema comun."""
    return [c for c in df_original.columns if c not in set(usadas)]


# Registro de que columnas se usaron en cada base (para el control de la seccion 12)
COLUMNAS_USADAS = {}

## 3. Guacamayos + Oyacachi: unificacion desde el formato ForestPlots

Los dos archivos vienen en el formato crudo de descarga de ForestPlots.net:

- **una hoja por parcela** (`GCM-06`, `GCM-07`, ..., `OYA-81`, ...);
- la **fila 1** trae los encabezados de bloque de censo (`Census No: 3  Date: 11/01/2023`)
  y la **fila 2** los nombres de campo (`D`, `POM`, `Flag1`, `Flag2`, `Census Notes`, ...);
- cada censo es un **bloque de columnas en horizontal**, y el numero de censos
  **varia entre parcelas** (Guacamayos tiene entre 1 y 3; Oyacachi entre 4 y 5);
- el DBH viene en **milimetros** (columna `D`), mientras que las otras bases estan
  en **centimetros** → se divide por 10;
- un mismo `Tag no` puede ocupar **varias filas** cuando el arbol es multi-fuste, asi
  que la clave de fila es la posicion en la hoja, no el tag.

El pipeline es: hojas → **tabla larga** (una fila por fuste × censo) → asignacion de
cada censo a las ranuras `censo1 / censo2 / censo3` → **tabla ancha** con el esquema comun.

### 3.1 Funciones de lectura del formato ForestPlots

In [3]:
CAMPO_FIN_BLOQUE = "Census Notes"        # ultima columna de cada bloque de censo
PATRON_HOJA_PLOT = r"^(GCM|OYA)[-_ ]?\d+$"  # hojas que son parcelas


def normaliza_plotid(txt):
    """'GCM-6' / 'GCM-06' / 'OYA-81' -> 'GCM_06' / 'OYA_81' (mismo formato que GAL_20, SUM_13)."""
    m = re.match(r"^\s*([A-Za-z]+)[-_ ]?(\d+)", str(txt))
    if not m:
        return np.nan
    return f"{m.group(1).upper()}_{int(m.group(2)):02d}"


def parse_encabezado_censo(txt):
    """'Census No: 2 (3?)  Date: 11/01/2023' -> (etiqueta, fecha, año, precision).

    Hay encabezados que traen solo el año ('Date: 2023') y otros con la numeracion
    dudosa ('Census No: 2 (3?)'). Se guarda el texto tal cual y se marca la precision.
    """
    txt = str(txt).strip()
    m = re.search(r"Census\s*No:\s*(.*?)\s*Date:", txt)
    etiqueta = m.group(1).strip() if m else ""
    m = re.search(r"Date:\s*(.+)$", txt)
    fecha_txt = m.group(1).strip() if m else ""
    if re.fullmatch(r"\d{4}", fecha_txt):          # solo el año
        return etiqueta, pd.NaT, int(fecha_txt), "solo_anio"
    f = pd.to_datetime(fecha_txt, errors="coerce")
    if pd.isna(f):
        return etiqueta, pd.NaT, np.nan, "sin_fecha"
    return etiqueta, f.normalize(), int(f.year), "dia"


def _col(bloque, *nombres):
    """Primera columna existente del bloque entre varios nombres alternativos."""
    for n in nombres:
        if n in bloque:
            return bloque[n]
    return None


def parse_hoja_forestplots(df_raw, hoja, site, archivo):
    """Convierte UNA hoja de parcela en formato largo (una fila por fuste x censo)."""
    fila_censos = df_raw.iloc[0]
    hdr   = [str(x).strip() for x in df_raw.iloc[1]]
    datos = df_raw.iloc[2:].reset_index(drop=True).dropna(how="all")
    fila_hoja = datos.index.to_numpy() + 3     # numero de fila en Excel (1-based)

    inicios = [j for j in range(df_raw.shape[1])
               if pd.notna(fila_censos.iloc[j]) and "Census" in str(fila_censos.iloc[j])]
    if not inicios:
        raise ValueError(f"{hoja}: no se encontraron bloques de censo en la fila 1")

    # columnas de metadatos del arbol: todo lo que esta antes del primer bloque de censo
    meta = {}
    for j in range(inicios[0]):
        meta.setdefault(hdr[j], datos.iloc[:, j])

    piezas = []
    for k, ini in enumerate(inicios):
        lim = inicios[k + 1] if k + 1 < len(inicios) else df_raw.shape[1]
        fin = lim
        for j in range(ini, lim):              # el bloque termina en 'Census Notes'
            if hdr[j] == CAMPO_FIN_BLOQUE:
                fin = j + 1
                break
        bloque = {}
        for j in range(ini, fin):
            bloque.setdefault(hdr[j], datos.iloc[:, j])

        etiqueta, fecha, anio, precision = parse_encabezado_censo(fila_censos.iloc[ini])

        # DBH: 'D' viene en mm; si solo existe 'D (original)' esa ya esta en cm
        s_mm = _col(bloque, "D", "D mm!!!", "D mm rounded")
        if s_mm is not None:
            dbh_cm = pd.to_numeric(s_mm, errors="coerce") / 10.0
        else:
            s_cm = _col(bloque, "D (original)", "D original")
            dbh_cm = (pd.to_numeric(s_cm, errors="coerce") if s_cm is not None
                      else pd.Series(np.nan, index=datos.index))
        dbh_cm = dbh_cm.replace(0, np.nan)     # 0 = arbol muerto/no medido, no es un diametro

        def opc(*n):
            s = _col(bloque, *n)
            return s.values if s is not None else np.nan

        sub = pd.DataFrame({
            "site": site, "hoja": hoja, "source_file": archivo, "fila_hoja": fila_hoja,
            "censo_orden": k + 1, "censo_etiqueta": etiqueta,
            "censo_encabezado": str(fila_censos.iloc[ini]),
            "date": fecha, "year": anio, "date_precision": precision,
            "dbh_cm": dbh_cm.values,
            "pom":   pd.to_numeric(pd.Series(opc("POM"), index=datos.index), errors="coerce").values,
            "flag1": opc("Flag1"), "flag2": opc("Flag2"),
            "census_notes": opc(CAMPO_FIN_BLOQUE),
        }, index=datos.index)

        for destino, origen in [("Plot_raw", "Plot"), ("tag_no", "Tag no"),
                                ("new_tag", "New Tag"), ("old_tag", "Old Tag"),
                                ("family", "Family"), ("nombre_cientifico", "Species"),
                                ("id_original", "original identification"),
                                ("new_identification", "New identification"),
                                ("tree_notes", "Tree Notes"), ("voucher_code", "voucher code")]:
            sub[destino] = meta[origen].values if origen in meta else np.nan
        piezas.append(sub)

    largo = pd.concat(piezas, ignore_index=True)
    largo = largo[largo["tag_no"].notna()].reset_index(drop=True)
    largo["PlotID"] = largo["Plot_raw"].fillna(largo["hoja"]).map(normaliza_plotid)
    # un arbol "tiene registro" en un censo si fue visitado: hay DBH, bandera o nota
    largo["tiene_registro"] = (largo["dbh_cm"].notna() | largo["flag1"].notna()
                               | largo["census_notes"].notna())
    return largo


def leer_forestplots(ruta, site, hojas_excluidas=()):
    """Lee todas las hojas-parcela de un archivo ForestPlots y devuelve la tabla larga."""
    xl = pd.ExcelFile(ruta)
    piezas, leidas, omitidas = [], [], []
    for hoja in xl.sheet_names:
        if hoja in hojas_excluidas or not re.match(PATRON_HOJA_PLOT, hoja):
            omitidas.append(hoja)
            continue
        piezas.append(parse_hoja_forestplots(xl.parse(hoja, header=None), hoja, site, Path(ruta).name))
        leidas.append(hoja)
    return pd.concat(piezas, ignore_index=True), leidas, omitidas

### 3.2 Lectura de las dos bases a formato largo

In [4]:
largos, resumen_hojas = [], []
for site, ruta in FILES_FP.items():
    lg, leidas, omitidas = leer_forestplots(ruta, site, HOJAS_EXCLUIDAS_FP)
    largos.append(lg)
    resumen_hojas.append({"site": site, "archivo": Path(ruta).name,
                          "n_parcelas": len(leidas), "parcelas": ", ".join(leidas),
                          "hojas_omitidas": ", ".join(omitidas) or "-"})
    print(f"{site:11s} -> {len(leidas)} parcelas, {lg.groupby(['hoja', 'fila_hoja']).ngroups} fustes, "
          f"{lg.shape[0]} filas fuste x censo   ({Path(ruta).name})")
    if omitidas:
        print(f"{'':11s}    hojas omitidas: {', '.join(omitidas)}")

largo_gcm_oya = pd.concat(largos, ignore_index=True)
print(f"\nTabla larga Guacamayos+Oyacachi: {largo_gcm_oya.shape[0]} filas x {largo_gcm_oya.shape[1]} columnas")

Guacamayos  -> 8 parcelas, 295 fustes, 765 filas fuste x censo   (GUACAMAYOS_ForestPlot_2025_jh.xlsx)
               hojas omitidas: headers_treedata
Oyacachi    -> 4 parcelas, 203 fustes, 885 filas fuste x censo   (Oyacachi_forestplot_2025_jh_GP.xlsx)
               hojas omitidas: OYA-81_GP

Tabla larga Guacamayos+Oyacachi: 1650 filas x 27 columnas


### 3.3 Diagnostico: que censos tiene cada parcela y a que ranura va cada uno

Como el numero de censos varia entre parcelas, hay que decidir **parcela por parcela**
que bloque corresponde a `censo1`, `censo2` y `censo3`. La asignacion se hace a nivel
de **campaña** (no de arbol) para que las fechas sean coherentes dentro de la parcela y
para que un recluta nuevo quede simplemente vacio en `censo1`, igual que en las bases de
Galeras / Selva Viva / Sumaco.

Reglas:

- `censo3` = ultimo censo con datos, **si** su año es >= `ANIO_RECENSO_MIN` (2023);
- `censo1` = primer censo con datos (linea base);
- `censo2` = el intermedio **mas reciente** que conserve al menos el `UMBRAL_INTERMEDIO`
  (80 %) de los DBH del mejor intermedio;
- los demas intermedios quedan como `extra`: no se pierden, se guardan en la tabla larga
  y se resumen en la columna `censos_intermedios_omitidos`.

In [5]:
def mapa_ranuras_por_plot(largo, anio_recenso_min=ANIO_RECENSO_MIN,
                          umbral_intermedio=UMBRAL_INTERMEDIO):
    """Asigna, parcela por parcela, cada bloque de censo a censo1 / censo2 / censo3 / extra."""
    filas = []
    for (site, plotid), g in largo.groupby(["site", "PlotID"], dropna=False):
        info = (g.groupby("censo_orden")
                 .agg(etiqueta=("censo_etiqueta", "first"),
                      encabezado=("censo_encabezado", "first"),
                      year=("year", "first"), date=("date", "first"),
                      precision=("date_precision", "first"),
                      n_registros=("tiene_registro", "sum"),
                      n_dbh=("dbh_cm", lambda s: s.notna().sum()))
                 .reset_index())
        # bloques que existen en el Excel pero estan completamente vacios
        info["bloque_vacio"] = info["n_registros"] == 0
        con_datos = info[~info["bloque_vacio"]].sort_values(["year", "censo_orden"],
                                                            na_position="last")
        info["ranura"] = np.where(info["bloque_vacio"], "vacio", "extra")

        if len(con_datos):
            ordenes = list(con_datos["censo_orden"])
            ultimo  = ordenes[-1]
            anio_ultimo = con_datos.loc[con_datos["censo_orden"] == ultimo, "year"].iloc[0]
            resto = ordenes
            if pd.notna(anio_ultimo) and anio_ultimo >= anio_recenso_min:
                info.loc[info["censo_orden"] == ultimo, "ranura"] = "censo3"
                resto = ordenes[:-1]
            if resto:
                info.loc[info["censo_orden"] == resto[0], "ranura"] = "censo1"
                intermedios = resto[1:]
                if intermedios:
                    cand = con_datos[con_datos["censo_orden"].isin(intermedios)]
                    tope = cand["n_dbh"].max()
                    buenos = cand[cand["n_dbh"] >= umbral_intermedio * tope]
                    elegido = buenos.sort_values(["year", "censo_orden"]).iloc[-1]["censo_orden"]
                    info.loc[info["censo_orden"] == elegido, "ranura"] = "censo2"
        info.insert(0, "site", site)
        info.insert(1, "PlotID", plotid)
        filas.append(info)
    return pd.concat(filas, ignore_index=True)


mapa_censos = mapa_ranuras_por_plot(largo_gcm_oya)
print(mapa_censos[["site", "PlotID", "censo_orden", "etiqueta", "year", "date",
                   "precision", "n_registros", "n_dbh", "ranura"]].to_string(index=False))

      site PlotID  censo_orden etiqueta  year       date precision  n_registros  n_dbh ranura
Guacamayos GCM_06            1        1  2006 2006-04-02       dia           28     28 censo1
Guacamayos GCM_06            2        2  2011 2011-05-02       dia           29     25 censo2
Guacamayos GCM_06            3        3  2023 2023-11-01       dia           38     27 censo3
Guacamayos GCM_07            1        1  2006 2006-06-15       dia           31     31 censo1
Guacamayos GCM_07            2        2  2011 2011-05-02       dia           31     20 censo2
Guacamayos GCM_07            3   2 (3?)  2023        NaT solo_anio           31     16 censo3
Guacamayos GCM_08            1        1  2006 2006-04-02       dia           23     23 censo1
Guacamayos GCM_08            2        2  2011 2011-05-02       dia           25     22 censo2
Guacamayos GCM_08            3        3  2023 2023-11-01       dia           27     15 censo3
Guacamayos GCM_27            1        1  2006 2006-04-01    

In [6]:
# --- Avisos sobre la estructura de censos de Guacamayos / Oyacachi ---
print("=" * 78)
print("AVISOS SOBRE LOS CENSOS DE GUACAMAYOS / OYACACHI")
print("=" * 78)

vacios = mapa_censos[mapa_censos["ranura"] == "vacio"]
if len(vacios):
    print("\n[1] Bloques de censo que existen en el Excel pero estan COMPLETAMENTE VACIOS")
    print("    (tienen fecha en el encabezado pero ni un solo DBH ni bandera):")
    for _, r in vacios.iterrows():
        print(f"    - {r['PlotID']}  bloque {int(r['censo_orden'])}  ({r['encabezado']})")
    print("    -> se ignoran: la fecha del encabezado NO se propaga a los arboles.")

sin_recenso = (mapa_censos[mapa_censos["ranura"] != "vacio"]
               .groupby(["site", "PlotID"])["ranura"]
               .apply(lambda s: "censo3" not in set(s)))
sin_recenso = sin_recenso[sin_recenso].index.tolist()
if sin_recenso:
    print(f"\n[2] Parcelas SIN recenso >= {ANIO_RECENSO_MIN} (se eliminan en la seccion 10):")
    for site, pid in sin_recenso:
        print(f"    - {pid} ({site})")

solo_anio = mapa_censos[(mapa_censos["precision"] == "solo_anio") &
                        (mapa_censos["ranura"].isin(["censo1", "censo2", "censo3"]))]
if len(solo_anio):
    print("\n[3] Censos cuyo encabezado trae SOLO EL AÑO (sin dia ni mes):")
    for _, r in solo_anio.iterrows():
        print(f"    - {r['PlotID']}  {r['ranura']}  ({r['encabezado']})")
    print("    -> se intenta imputar la fecha exacta en la seccion 9; si no se puede,")
    print("       queda la columna year_censusN con el año.")

dup_fecha = (mapa_censos[mapa_censos["ranura"] != "vacio"]
             .dropna(subset=["date"])
             .groupby(["PlotID", "date"]).size())
dup_fecha = dup_fecha[dup_fecha > 1]
if len(dup_fecha):
    print("\n[4] Parcelas con DOS bloques de censo con la MISMA fecha (revisar en el Excel):")
    for (pid, f), n in dup_fecha.items():
        print(f"    - {pid}: {n} bloques con fecha {f.date()}")

AVISOS SOBRE LOS CENSOS DE GUACAMAYOS / OYACACHI

[1] Bloques de censo que existen en el Excel pero estan COMPLETAMENTE VACIOS
    (tienen fecha en el encabezado pero ni un solo DBH ni bandera):
    - GCM_27  bloque 2  (Census No: 2  Date: 5/2/2011)
    - GCM_27  bloque 3  (Census No: 2 (3?)  Date: 2023)
    - GCM_52  bloque 2  (Census No: 1 (2?)  Date: 2011)
    - GCM_52  bloque 3  (Census No: 2 (3?)  Date: 2023)
    - GCM_53  bloque 2  (Census No: 1 (2?)  Date: 2011)
    - GCM_53  bloque 3  (Census No: 2 (3?)  Date: 2023)
    -> se ignoran: la fecha del encabezado NO se propaga a los arboles.

[2] Parcelas SIN recenso >= 2023 (se eliminan en la seccion 10):
    - GCM_27 (Guacamayos)
    - GCM_52 (Guacamayos)
    - GCM_53 (Guacamayos)
    - GCM_54 (Guacamayos)

[3] Censos cuyo encabezado trae SOLO EL AÑO (sin dia ni mes):
    - GCM_07  censo3  (Census No: 2 (3?)  Date: 2023)
    - OYA_83  censo1  (Census No: 1  Date: 2006)
    -> se intenta imputar la fecha exacta en la seccion 9; si 

### 3.4 Pivote de N censos al esquema de 3 censos

In [7]:
CLAVE_FUSTE = ["source_file", "hoja", "fila_hoja"]


def clasifica_muerto(flag1, flag2):
    """Traduce las banderas de ForestPlots a 1 = muerto, 0 = vivo, NaN = indefinido.

    Flag1 = '0' -> el arbol no esta vivo;  Flag1 empieza con 'a' -> vivo.
    Flag2 empieza con 'k' -> registrado como muerto;  Flag2 = '1' -> vivo y encontrado.
    """
    f1 = "" if pd.isna(flag1) else str(flag1).strip().lower()
    f2 = "" if pd.isna(flag2) else str(flag2).strip().lower()
    if f1 == "0" or f2.startswith("k"):
        return 1.0
    if f1.startswith("a") or f2 == "1":
        return 0.0
    return np.nan


def separa_genero_especie(nombre):
    """'Palicourea stenosepala' -> ('Palicourea', 'stenosepala'); 'Licania' -> ('Licania', NaN).

    Las otras bases guardan el genero y el epiteto por separado, aqui vienen juntos.
    """
    if pd.isna(nombre):
        return pd.Series([np.nan, np.nan])
    partes = str(nombre).strip().split()
    if not partes:
        return pd.Series([np.nan, np.nan])
    if len(partes) == 1:
        return pd.Series([partes[0], np.nan])
    return pd.Series([partes[0], " ".join(partes[1:])])


def pivota_a_tres_censos(largo, mapa):
    """Tabla larga (fuste x censo) -> tabla ancha (un fuste por fila, 3 censos)."""
    largo = largo.merge(mapa[["site", "PlotID", "censo_orden", "ranura"]],
                        on=["site", "PlotID", "censo_orden"], how="left")
    largo["ranura"] = largo["ranura"].fillna("vacio")
    largo["dead"] = [clasifica_muerto(a, b) for a, b in zip(largo["flag1"], largo["flag2"])]

    base = (largo.sort_values(CLAVE_FUSTE + ["censo_orden"])
                 .drop_duplicates(CLAVE_FUSTE)
                 .set_index(CLAVE_FUSTE))
    ancho = base[["site", "PlotID", "tag_no", "new_tag", "old_tag", "family",
                  "nombre_cientifico", "tree_notes", "voucher_code"]].copy()

    for r in ["censo1", "censo2", "censo3"]:
        n = r[-1]
        sub = largo[largo["ranura"] == r].drop_duplicates(CLAVE_FUSTE).set_index(CLAVE_FUSTE)
        ancho[f"date_census{n}"]      = sub["date"]
        ancho[f"year_census{n}"]      = sub["year"]
        ancho[f"precision_census{n}"] = sub["date_precision"]
        ancho[f"dbh_census{n}"]       = sub["dbh_cm"]
        ancho[f"dead_census{n}"]      = sub["dead"]
        ancho[f"comment_census{n}"]   = sub["census_notes"]
        # un arbol que no fue visitado en esa campaña no debe heredar la fecha de la parcela
        no_visitado = (ancho[f"dbh_census{n}"].isna() & ancho[f"dead_census{n}"].isna()
                       & ancho[f"comment_census{n}"].isna())
        ancho.loc[no_visitado, [f"date_census{n}", f"year_census{n}", f"precision_census{n}"]] = np.nan

    def _texto_extra(f):
        anio = "s/f" if pd.isna(f["year"]) else str(int(f["year"]))
        dbh  = "sin dbh" if pd.isna(f["dbh_cm"]) else f"dbh={f['dbh_cm']:.1f}"
        return f"censo {f['censo_etiqueta']} ({anio}): {dbh}"

    df_extra = largo[(largo["ranura"] == "extra") & largo["tiene_registro"]].copy()
    if len(df_extra):
        df_extra["txt"] = df_extra.apply(_texto_extra, axis=1)
        ancho["censos_intermedios_omitidos"] = df_extra.groupby(CLAVE_FUSTE)["txt"].apply(" | ".join)
    else:
        ancho["censos_intermedios_omitidos"] = np.nan
    ancho["n_censos_con_registro"] = largo[largo["tiene_registro"]].groupby(CLAVE_FUSTE).size()
    return ancho.reset_index()


gcm_oya_ancho = pivota_a_tres_censos(largo_gcm_oya, mapa_censos)
gcm_oya_ancho[["genus", "species"]] = gcm_oya_ancho["nombre_cientifico"].apply(separa_genero_especie)
gcm_oya_ancho["Plot"] = gcm_oya_ancho["PlotID"].str.extract(r"(\d+)$").astype(float)
# fustes por tag: >1 indica arbol multi-fuste (varias filas con el mismo Tag no)
gcm_oya_ancho["n_fustes_tag"] = gcm_oya_ancho.groupby(["PlotID", "tag_no"])["fila_hoja"].transform("size")

print(f"Guacamayos + Oyacachi: {gcm_oya_ancho.shape[0]} fustes "
      f"({gcm_oya_ancho['tag_no'].nunique()} tags unicos) x {gcm_oya_ancho.shape[1]} columnas")
multi = gcm_oya_ancho[gcm_oya_ancho["n_fustes_tag"] > 1]
print(f"Arboles multi-fuste: {multi['tag_no'].nunique()} tags con {len(multi)} filas en total")
gcm_oya_ancho.head(3)

Guacamayos + Oyacachi: 498 fustes (450 tags unicos) x 36 columnas
Arboles multi-fuste: 29 tags con 76 filas en total


,source_file,hoja,fila_hoja,site,PlotID,tag_no,new_tag,old_tag,family,nombre_cientifico,tree_notes,voucher_code,date_census1,year_census1,precision_census1,dbh_census1,dead_census1,comment_census1,date_census2,year_census2,precision_census2,dbh_census2,dead_census2,comment_census2,date_census3,year_census3,precision_census3,dbh_census3,dead_census3,comment_census3,censos_intermedios_omitidos,n_censos_con_registro,genus,species,Plot,n_fustes_tag
0,GUACAMAYOS_ForestPlot_2025_jh.xlsx,GCM-06,3,Guacamayos,GCM_06,142,NaN,142,Rubiaceae,Palicourea stenosepala,NaN,NaN,2006-04-02,2006.0,dia,12.42,0.0,NaN,2011-05-02,2011.0,dia,12.414,0.0,NaN,2023-11-01,2023.0,dia,NaN,1.0,NaN,NaN,3.0,Palicourea,stenosepala,6.0,1
1,GUACAMAYOS_ForestPlot_2025_jh.xlsx,GCM-06,4,Guacamayos,GCM_06,143,8203,143,Cunoniaceae,Weinmannia pinnata,NaN,NaN,2006-04-02,2006.0,dia,24.35,0.0,NaN,2011-05-02,2011.0,dia,26.550,0.0,NaN,2023-11-01,2023.0,dia,30.8,0.0,8.8 m was the distance at which the height of ...,NaN,3.0,Weinmannia,pinnata,6.0,1
2,GUACAMAYOS_ForestPlot_2025_jh.xlsx,GCM-06,5,Guacamayos,GCM_06,144,8204,144,Adoxaceae,Viburnum,NaN,NaN,2006-04-02,2006.0,dia,19.28,0.0,NaN,2011-05-02,2011.0,dia,20.213,0.0,NaN,2023-11-01,2023.0,dia,20.9,0.0,7.9 m was the distance at which the height of ...,NaN,3.0,Viburnum,NaN,6.0,1


### 3.5 Mapeo al esquema comun

In [8]:
def standardize_gcm_oya(df):
    """Guacamayos + Oyacachi (ya unificados) -> esquema comun CORE_COLUMNS."""
    df = df.copy()
    out = empty_core_df(len(df))

    out["dataset_source"] = "Guacamayos_Oyacachi"
    out["Site"]           = df["site"].values          # Guacamayos u Oyacachi
    out["PlotID"]         = df["PlotID"].values
    out["Plot"]           = df["Plot"].values
    # estas hojas no traen subplot (las columnas T1/T2 estan vacias)
    out["treeID"]         = df["tag_no"].values        # 'Tag no' = ID usado en el censo
    out["Old_Tag"]        = df["old_tag"].values       # etiqueta original
    out["new_tree_ID"]    = df["new_tag"].values       # etiqueta nueva
    out["family"]         = df["family"].values
    out["genus"]          = df["genus"].values
    out["species"]        = df["species"].values

    for n in (1, 2, 3):
        out[f"date_census{n}"]    = df[f"date_census{n}"].values
        out[f"year_census{n}"]    = df[f"year_census{n}"].values
        out[f"dbh_census{n}"]     = df[f"dbh_census{n}"].values
        out[f"dead_census{n}"]    = df[f"dead_census{n}"].values
        out[f"comment_census{n}"] = df[f"comment_census{n}"].values

    # las notas generales del arbol y los censos intermedios que no entraron en el esquema
    # de 3 censos van en columnas propias: si se metieran en comment_census1 se veria como
    # si el arbol hubiera sido visitado en el censo 1
    out["tree_notes"]         = df["tree_notes"].values
    out["censos_intermedios"] = df["censos_intermedios_omitidos"].values
    out["comment_taxo"]       = df["nombre_cientifico"].values   # nombre tal cual venia en el Excel
    out["source_file"]    = df["source_file"].values
    return out


df_gcm_oya = standardize_gcm_oya(gcm_oya_ancho)
COLUMNAS_USADAS["Guacamayos_Oyacachi"] = [
    "Plot", "Tag no", "New Tag", "Old Tag", "Family", "Species", "Tree Notes",
    "D", "D mm!!!", "D (original)", "D original", "Flag1", "Flag2", "Census Notes"]
print(f"df_gcm_oya: {df_gcm_oya.shape[0]} filas x {df_gcm_oya.shape[1]} columnas")
df_gcm_oya.head(3)

df_gcm_oya: 498 filas x 30 columnas


,dataset_source,Site,PlotID,Plot,Subplot,treeID,Old_Tag,new_tree_ID,family,genus,species,date_census1,year_census1,dbh_census1,dead_census1,comment_census1,date_census2,year_census2,dbh_census2,dead_census2,comment_census2,date_census3,year_census3,dbh_census3,dead_census3,comment_census3,tree_notes,censos_intermedios,comment_taxo,source_file
0,Guacamayos_Oyacachi,Guacamayos,GCM_06,6.0,NaN,142,142,NaN,Rubiaceae,Palicourea,stenosepala,2006-04-02,2006.0,12.42,0.0,NaN,2011-05-02,2011.0,12.414,0.0,NaN,2023-11-01,2023.0,NaN,1.0,NaN,NaN,NaN,Palicourea stenosepala,GUACAMAYOS_ForestPlot_2025_jh.xlsx
1,Guacamayos_Oyacachi,Guacamayos,GCM_06,6.0,NaN,143,143,8203,Cunoniaceae,Weinmannia,pinnata,2006-04-02,2006.0,24.35,0.0,NaN,2011-05-02,2011.0,26.550,0.0,NaN,2023-11-01,2023.0,30.8,0.0,8.8 m was the distance at which the height of ...,NaN,NaN,Weinmannia pinnata,GUACAMAYOS_ForestPlot_2025_jh.xlsx
2,Guacamayos_Oyacachi,Guacamayos,GCM_06,6.0,NaN,144,144,8204,Adoxaceae,Viburnum,NaN,2006-04-02,2006.0,19.28,0.0,NaN,2011-05-02,2011.0,20.213,0.0,NaN,2023-11-01,2023.0,20.9,0.0,7.9 m was the distance at which the height of ...,NaN,NaN,Viburnum,GUACAMAYOS_ForestPlot_2025_jh.xlsx


### 3.6 Guardar la base intermedia Guacamayos + Oyacachi

Esta base queda guardada aparte para poder revisarla o reutilizarla sin volver a
correr todo. El archivo tiene tres hojas:

- `Datos_unificados`: las dos parcelas ya en el esquema comun (lo que se concatena despues);
- `Detalle_completo`: la misma tabla pero con las columnas extra (banderas, precision de
  fechas, censos intermedios, n_fustes);
- `Mapa_censos`: que bloque de censo de cada parcela fue a que ranura.

In [9]:
with pd.ExcelWriter(OUT_GCM_OYA, engine="openpyxl") as xw:
    df_gcm_oya.to_excel(xw, sheet_name="Datos_unificados", index=False)
    gcm_oya_ancho.to_excel(xw, sheet_name="Detalle_completo", index=False)
    mapa_censos.to_excel(xw, sheet_name="Mapa_censos", index=False)
    pd.DataFrame(resumen_hojas).to_excel(xw, sheet_name="Hojas_leidas", index=False)

print(f"Guardado: {OUT_GCM_OYA}")
print("Hojas: Datos_unificados | Detalle_completo | Mapa_censos | Hojas_leidas")

Guardado: Base_unificada_ForestPlots_Guacamayos_Oyacachi.xlsx
Hojas: Datos_unificados | Detalle_completo | Mapa_censos | Hojas_leidas


## 4. Cargar las otras tres bases (Galeras, Selva Viva, Sumaco)

In [10]:
raw = {}
for name, path in FILES.items():
    raw[name] = pd.read_excel(path, sheet_name=SHEETS[name])
    print(f"{name:12s} -> {raw[name].shape[0]:4d} filas, {raw[name].shape[1]:3d} columnas   ({path.name})")

Galeras      ->  636 filas,  40 columnas   (Base_Galeras_vK.xlsx)
SelvaViva    ->  354 filas,  33 columnas   (Base_SelvaViva_v2.xlsx)


Sumaco       ->  625 filas,  33 columnas   (Sumaco_17092025.xlsx)


## 5. Homogeneizar Galeras

In [11]:
def standardize_galeras(df):
    df = df.copy()
    out = empty_core_df(len(df))

    out["dataset_source"] = "Galeras"
    out["Site"]           = df["site"]
    out["PlotID"]         = df["PlotID"]
    out["Plot"]           = df["Plot"]
    # esta base no tiene columna de subplot
    out["treeID"]         = df["treeID"]
    # esta base no tiene columna de ID viejo tipo "Old_Tag"
    out["new_tree_ID"]    = df["new ID 2024"]
    out["family"]         = df["family"]
    out["genus"]          = df["genus"]
    out["species"]        = df["species"]

    out["date_census1"]    = df["date census 1"]
    out["dbh_census1"]     = df["dbh 1"]
    out["comment_census1"] = df["comment"]

    out["date_census2"]    = df["date census 2"]
    out["dbh_census2"]     = df["dbh 2"]
    out["dead_census2"]    = df["dead census 2"]

    out["date_census3"]    = df["date census 3"]
    out["dbh_census3"]     = df["dbh 2024"]
    out["dead_census3"]    = df["dead 2024"]
    out["comment_census3"] = df["comment 2024"]

    out["comment_taxo"]   = df["comment taxo 2024"]
    out["source_file"]    = "Base_Galeras_vK.xlsx"
    return out


COLUMNAS_USADAS["Galeras"] = [
    "site", "PlotID", "Plot", "treeID", "new ID 2024", "family", "genus", "species",
    "date census 1", "dbh 1", "comment", "date census 2", "dbh 2", "dead census 2",
    "date census 3", "dbh 2024", "dead 2024", "comment 2024", "comment taxo 2024"]

df_galeras = standardize_galeras(raw["Galeras"])
df_galeras.head(3)

,dataset_source,Site,PlotID,Plot,Subplot,treeID,Old_Tag,new_tree_ID,family,genus,species,date_census1,year_census1,dbh_census1,dead_census1,comment_census1,date_census2,year_census2,dbh_census2,dead_census2,comment_census2,date_census3,year_census3,dbh_census3,dead_census3,comment_census3,tree_notes,censos_intermedios,comment_taxo,source_file
0,Galeras,Galeras,GAL_20,20,NaN,471,NaN,NaN,Rubiaceae,Randia,NaN,2006-04-15,NaN,17.55,NaN,NaN,2011-06-30,NaN,"18,43",0.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Base_Galeras_vK.xlsx
1,Galeras,Galeras,GAL_20,20,NaN,472,NaN,NaN,Rubiaceae,Randia,NaN,2006-04-15,NaN,13.65,NaN,NaN,2011-06-30,NaN,"13,80",0.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Base_Galeras_vK.xlsx
2,Galeras,Galeras,GAL_20,20,NaN,473,NaN,NaN,Burseraceae,Dacryodes,aff cupularis,2006-04-15,NaN,12.26,NaN,NaN,2011-06-30,NaN,"13,42",0.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Base_Galeras_vK.xlsx


## 6. Homogeneizar Selva Viva

In [12]:
def standardize_selvaviva(df):
    df = df.copy()
    out = empty_core_df(len(df))

    out["dataset_source"] = "SelvaViva"
    out["Site"]           = df["site"]
    out["PlotID"]         = df["PlotID"]
    # no hay columna de numero de plot -> se extrae desde PlotID (ej. SEV_68 -> 68)
    out["Plot"]           = df["PlotID"].str.extract(r"(\d+)$").astype(float)
    out["Subplot"]        = df["subplot"]
    out["treeID"]         = df["treeID"]
    out["new_tree_ID"]    = df["new ID 2024"]
    out["family"]         = df["family"]
    out["genus"]          = df["genus"]
    out["species"]        = df["species"]

    out["date_census1"]    = df["date census 1"]
    out["dbh_census1"]     = df["dbh 1"]
    out["comment_census1"] = df["comment"]

    out["date_census2"]    = df["date census 2"]
    out["dbh_census2"]     = df["dbh 2"]
    out["dead_census2"]    = df["dead census 2"]

    out["date_census3"]    = df["date census 3 2025"]
    out["dbh_census3"]     = df["dbh 3 2025"]
    out["dead_census3"]    = df["dead 2025"]
    out["comment_census3"] = df["comment 2024"]

    out["comment_taxo"]   = df["comment taxo 2024"]
    out["source_file"]    = "Base_SelvaViva_v2.xlsx"
    return out


COLUMNAS_USADAS["SelvaViva"] = [
    "site", "PlotID", "subplot", "treeID", "new ID 2024", "family", "genus", "species",
    "date census 1", "dbh 1", "comment", "date census 2", "dbh 2", "dead census 2",
    "date census 3 2025", "dbh 3 2025", "dead 2025", "comment 2024", "comment taxo 2024"]

df_selvaviva = standardize_selvaviva(raw["SelvaViva"])
df_selvaviva.head(3)

,dataset_source,Site,PlotID,Plot,Subplot,treeID,Old_Tag,new_tree_ID,family,genus,species,date_census1,year_census1,dbh_census1,dead_census1,comment_census1,date_census2,year_census2,dbh_census2,dead_census2,comment_census2,date_census3,year_census3,dbh_census3,dead_census3,comment_census3,tree_notes,censos_intermedios,comment_taxo,source_file
0,SelvaViva,Selva Viva,SEV_68,68.0,A,2146,NaN,8865.0,Violaceae,Leonia,glycycarpa,2007-03-28,NaN,21.326762,NaN,"VIO, Leonia glycycarpa Ruiz & Pav. Edison",NaT,NaN,NaN,NaN,NaN,2025-03-03,NaN,24.2,NaN,Next to a dead standing tree,NaN,NaN,NaN,Base_SelvaViva_v2.xlsx
1,SelvaViva,Selva Viva,SEV_68,68.0,A,2144,NaN,8866.0,Sapotaceae,Chrysophyllum,argenteum,2007-03-28,NaN,33.740848,NaN,"SAPO, Chrysophyllum argenteum Jacq. Miguel",NaT,NaN,NaN,NaN,NaN,2025-02-03,NaN,33.7,NaN,pv 2144,NaN,NaN,NaN,Base_SelvaViva_v2.xlsx
2,SelvaViva,Selva Viva,SEV_68,68.0,A,2142,NaN,8867.0,Rhizophoraceae,Cassipourea,guianensis,2007-03-28,NaN,13.687325,NaN,RHI,NaT,NaN,NaN,NaN,NaN,2025-02-03,NaN,14.4,NaN,pv 2142,NaN,NaN,NaN,Base_SelvaViva_v2.xlsx


## 7. Homogeneizar Sumaco

In [13]:
def standardize_sumaco(df):
    df = df.copy()
    out = empty_core_df(len(df))

    out["dataset_source"] = "Sumaco"
    out["Site"]           = "Sumaco"
    out["PlotID"]         = df["plot ID"]
    # se extrae el numero de plot desde el PlotID (ej. SUM_09 -> 9)
    out["Plot"]           = df["plot ID"].str.extract(r"(\d+)$").astype(float)
    out["Subplot"]        = df["Subplot"]
    out["treeID"]         = df["treeID"]
    out["new_tree_ID"]    = df["new tree ID 2025"]
    out["family"]         = df["family"]
    out["genus"]          = df["genus"]
    out["species"]        = df["species"]

    out["date_census1"]    = df["date census 1"]
    out["dbh_census1"]     = df["dbh 1"]
    out["comment_census1"] = df["comment"]

    out["date_census2"]    = df["date census 2"]
    out["dbh_census2"]     = df["dbh 2"]
    out["dead_census2"]    = df["dead census 2"]

    out["date_census3"]    = df["date census 3"]
    out["dbh_census3"]     = df["dbh 3"]
    out["dead_census3"]    = df["dead 2025"]
    out["comment_census3"] = df["comment 2024"]

    out["comment_taxo"]   = df["comment taxo"]
    out["source_file"]    = "Sumaco_17092025.xlsx"
    return out


COLUMNAS_USADAS["Sumaco"] = [
    "plot ID", "Subplot", "treeID", "new tree ID 2025", "family", "genus", "species",
    "date census 1", "dbh 1", "comment", "date census 2", "dbh 2", "dead census 2",
    "date census 3", "dbh 3", "dead 2025", "comment 2024", "comment taxo"]

df_sumaco = standardize_sumaco(raw["Sumaco"])
df_sumaco.head(3)

,dataset_source,Site,PlotID,Plot,Subplot,treeID,Old_Tag,new_tree_ID,family,genus,species,date_census1,year_census1,dbh_census1,dead_census1,comment_census1,date_census2,year_census2,dbh_census2,dead_census2,comment_census2,date_census3,year_census3,dbh_census3,dead_census3,comment_census3,tree_notes,censos_intermedios,comment_taxo,source_file
0,Sumaco,Sumaco,SUM_09,9.0,NaN,224,NaN,NaN,Moraceae,Ficus,quijosana,2006-04-06,NaN,47.74,NaN,NaN,2011-05-12,NaN,48.990000,0.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sumaco_17092025.xlsx
1,Sumaco,Sumaco,SUM_09,9.0,NaN,225,NaN,NaN,Moraceae,Ficus,quijosana,2006-04-06,NaN,45.97,NaN,NaN,2011-05-12,NaN,47.370000,0.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sumaco_17092025.xlsx
2,Sumaco,Sumaco,SUM_09,9.0,NaN,226,NaN,NaN,indet,NaN,NaN,2006-04-06,NaN,34,NaN,NaN,2011-05-12,NaN,34.218313,0.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sumaco_17092025.xlsx


## 8. Concatenar todas las bases homogeneizadas

In [14]:
df_todo = pd.concat(
    [df_gcm_oya, df_galeras, df_selvaviva, df_sumaco],
    ignore_index=True,
)[CORE_COLUMNS]

# tipos de datos consistentes
for c in COLS_DATE:
    df_todo[c] = pd.to_datetime(df_todo[c], errors="coerce")
for c in COLS_DBH + COLS_YEAR + ["Plot"]:
    df_todo[c] = pd.to_numeric(df_todo[c], errors="coerce")
for c in ["dead_census1", "dead_census2", "dead_census3"]:
    df_todo[c] = pd.to_numeric(df_todo[c], errors="coerce")

# un DBH de 0 no es una medida (suele marcar un arbol muerto o no medido) -> NaN
n_ceros = int((df_todo[COLS_DBH] == 0).sum().sum())
df_todo[COLS_DBH] = df_todo[COLS_DBH].replace(0, np.nan)
if n_ceros:
    print(f"[i] {n_ceros} valores de DBH = 0 convertidos a NaN\n")

print(f"Base combinada (antes de homogenizar fechas y filtrar): "
      f"{df_todo.shape[0]} filas x {df_todo.shape[1]} columnas")
print()
print(df_todo.groupby("dataset_source").size().to_frame("n_filas").to_string())

[i] 3 valores de DBH = 0 convertidos a NaN

Base combinada (antes de homogenizar fechas y filtrar): 2113 filas x 30 columnas

                     n_filas
dataset_source              
Galeras                  636
Guacamayos_Oyacachi      498
SelvaViva                354
Sumaco                   625


## 9. Homogeneizar las fechas y avisar de las que faltan

Todas las fechas quedan como `datetime64` sin hora. Cuando un arbol **tiene dato en un
censo** (DBH, bandera de muerte o comentario) pero **no tiene fecha**, se imputa en tres
niveles, del mas especifico al mas general:

1. **`moda_parcela`** — fecha mas frecuente de ese mismo censo en la misma parcela;
2. **`moda_sitio_anio`** — si se conoce el año pero no el dia (caso de los encabezados
   `Date: 2023` de Guacamayos), fecha mas frecuente registrada en ese sitio ese año;
3. **`moda_base_censo`** — fecha mas frecuente de esa campaña en toda la base.

La columna `date_censusN_origen` deja registrado de donde salio cada fecha:
`original`, una de las tres imputaciones, `solo_anio` (solo se pudo recuperar el año) o
`FALTA` (no se pudo determinar nada). Los arboles que no fueron visitados en ese censo
quedan como `sin_censo` y **no** reciben fecha.

In [15]:
ETIQUETAS_ORIGEN = ["original", "moda_parcela", "moda_sitio_anio",
                    "moda_base_censo", "solo_anio", "FALTA"]


def _moda(s):
    """Valor mas frecuente de una serie de fechas (NaT si no hay ninguna)."""
    m = s.dropna().mode()
    return m.iloc[0] if len(m) else pd.NaT


def homogeneiza_fechas(df):
    """Normaliza a datetime y completa las fechas de los 3 censos.

    Agrega date_censusN_origen con la procedencia de cada fecha. Nunca mezcla
    campañas de años distintos: si se conoce el año (encabezados de ForestPlots que
    solo traen 'Date: 2023'), solo se acepta una referencia de ese mismo año.
    """
    df = df.copy()

    # referencia sitio-año -> fecha modal (mirando los 3 censos a la vez)
    ref = pd.concat([df[["Site", c]].rename(columns={c: "f"}) for c in COLS_DATE]).dropna()
    ref["anio"] = ref["f"].dt.year
    moda_sitio_anio = ref.groupby(["Site", "anio"])["f"].agg(_moda)

    for n in (1, 2, 3):
        col_f, col_a = f"date_census{n}", f"year_census{n}"
        df[col_f] = pd.to_datetime(df[col_f], errors="coerce").dt.normalize()
        anio_prev = pd.to_numeric(df[col_a], errors="coerce")   # año que venia del encabezado

        # "visitado" = hay evidencia dura de que el arbol se midio en esa campaña.
        # Los comentarios NO cuentan: muchos son notas generales del arbol y harian
        # que se le invente una fecha a un censo en el que nunca se lo midio.
        visitado = (df[f"dbh_census{n}"].notna() | df[f"dead_census{n}"].notna()
                    | df[col_f].notna() | anio_prev.notna())

        origen = pd.Series(np.nan, index=df.index, dtype="object")
        origen[df[col_f].notna()] = "original"

        moda_plot = df.groupby(["dataset_source", "PlotID"])[col_f].transform(_moda)
        moda_base = df.groupby("dataset_source")[col_f].transform(_moda)
        clave = list(zip(df["Site"], anio_prev))
        moda_sa = pd.to_datetime(
            pd.Series([moda_sitio_anio.get(k, pd.NaT) for k in clave], index=df.index),
            errors="coerce")

        # (1) moda de la parcela, siempre que sea coherente con el año conocido
        coherente = anio_prev.isna() | (moda_plot.dt.year == anio_prev)
        falta = df[col_f].isna() & visitado & moda_plot.notna() & coherente
        df.loc[falta, col_f] = moda_plot[falta]
        origen[falta] = "moda_parcela"

        # (2) moda del sitio en ese año (para los encabezados que solo traen el año)
        falta = df[col_f].isna() & visitado & moda_sa.notna()
        df.loc[falta, col_f] = moda_sa[falta]
        origen[falta] = "moda_sitio_anio"

        # (3) moda de la campaña en toda la base: solo cuando no se conoce el año,
        #     para no arrastrar la fecha de un sitio a otro
        falta = df[col_f].isna() & visitado & moda_base.notna() & anio_prev.isna()
        df.loc[falta, col_f] = moda_base[falta]
        origen[falta] = "moda_base_censo"

        # año final: el de la fecha si existe, si no el que ya traia el encabezado
        df[col_a] = df[col_f].dt.year.fillna(anio_prev).astype("Int64")

        origen[df[col_f].isna() & visitado & df[col_a].notna()] = "solo_anio"
        origen[df[col_f].isna() & visitado & df[col_a].isna()] = "FALTA"
        origen[~visitado] = "sin_censo"
        df[f"date_census{n}_origen"] = origen

    return df


df_todo = homogeneiza_fechas(df_todo)

resumen_origen = pd.DataFrame({
    f"censo{n}": df_todo[f"date_census{n}_origen"].value_counts() for n in (1, 2, 3)
}).fillna(0).astype(int)
print("Origen de las fechas (numero de filas):")
print(resumen_origen.to_string())

Origen de las fechas (numero de filas):
                 censo1  censo2  censo3
moda_parcela          0       1       5
moda_sitio_anio       0       0      31
original           1839     691     978
sin_censo           267    1421    1099
solo_anio             7       0       0


In [16]:
# ------------------- AVISO DE FECHAS FALTANTES O IMPUTADAS -------------------
filas = []
for n in (1, 2, 3):
    g = (df_todo.groupby(["dataset_source", "PlotID"])[f"date_census{n}_origen"]
         .value_counts().unstack(fill_value=0).reset_index())
    for etiqueta in ETIQUETAS_ORIGEN:
        if etiqueta not in g.columns:
            g[etiqueta] = 0
    fechas = (df_todo.groupby(["dataset_source", "PlotID"])[f"date_census{n}"].max()
              .reset_index().rename(columns={f"date_census{n}": "fecha"}))
    g = g.merge(fechas, on=["dataset_source", "PlotID"], how="left")
    g.insert(2, "censo", n)
    filas.append(g[["dataset_source", "PlotID", "censo", "fecha"] + ETIQUETAS_ORIGEN])

reporte_fechas = (pd.concat(filas, ignore_index=True)
                  .sort_values(["dataset_source", "PlotID", "censo"])
                  .reset_index(drop=True))
reporte_fechas["n_imputadas"] = reporte_fechas[
    ["moda_parcela", "moda_sitio_anio", "moda_base_censo"]].sum(axis=1)

print("=" * 78)
print("AVISO DE FECHAS")
print("=" * 78)

pendientes = reporte_fechas[(reporte_fechas["FALTA"] > 0) | (reporte_fechas["solo_anio"] > 0)]
if len(pendientes) == 0:
    print("\nOK: todas las filas con dato de censo quedaron con fecha completa.")
else:
    print("\n[!] Parcelas/censos donde NO se pudo reconstruir la fecha completa:")
    print(pendientes[["dataset_source", "PlotID", "censo", "solo_anio", "FALTA"]]
          .to_string(index=False))
    print("\n    'solo_anio' = se conoce el año (queda en year_censusN) pero no el dia.")
    print("    'FALTA'     = no hay ninguna referencia de fecha para esas filas.")

imputadas = reporte_fechas[reporte_fechas["n_imputadas"] > 0]
if len(imputadas):
    print("\n[i] Parcelas/censos con fechas imputadas (revisar si el valor es razonable):")
    print(imputadas[["dataset_source", "PlotID", "censo", "fecha", "moda_parcela",
                     "moda_sitio_anio", "moda_base_censo"]].to_string(index=False))

print("\n[i] Rango de fechas por base y censo:")
rango = pd.DataFrame({
    f"censo{n}": df_todo.groupby("dataset_source")[f"date_census{n}"]
                        .agg(lambda s: "-" if s.isna().all()
                             else f"{s.min().date()} a {s.max().date()}")
    for n in (1, 2, 3)})
print(rango.to_string())

AVISO DE FECHAS

[!] Parcelas/censos donde NO se pudo reconstruir la fecha completa:
     dataset_source PlotID  censo  solo_anio  FALTA
Guacamayos_Oyacachi OYA_83      1          7      0

    'solo_anio' = se conoce el año (queda en year_censusN) pero no el dia.
    'FALTA'     = no hay ninguna referencia de fecha para esas filas.

[i] Parcelas/censos con fechas imputadas (revisar si el valor es razonable):
     dataset_source PlotID  censo      fecha  moda_parcela  moda_sitio_anio  moda_base_censo
            Galeras GAL_48      3 2025-01-09             2                0                0
            Galeras GAL_50      3 2024-09-05             3                0                0
Guacamayos_Oyacachi GCM_07      3 2023-11-01             0               31                0
          SelvaViva SEV_74      2 2011-04-28             1                0                0

[i] Rango de fechas por base y censo:
                                      censo1                   censo2              

## 10. Filtrar: conservar los plots con recenso reciente (2023/2024/2025)

Un **plot** se considera recensado si **al menos uno** de sus arboles tiene DBH o
bandera de mortalidad en el censo 3. Los plots donde ningun arbol fue revisitado se
eliminan, **excepto** los de `PLOTS_FORZADOS` (`GAL_20`, `GAL_29`, `SUM_13`), que se
conservan porque hacen falta para el analisis posterior y quedan marcados con
`recenso_forzado = True`.

In [17]:
# un arbol tiene censo 3 si fue medido o si quedo registrado como muerto en esa campaña
tiene_censo3 = df_todo["dbh_census3"].notna() | df_todo["dead_census3"].notna()

resumen_plots = (df_todo.assign(_c3=tiene_censo3)
                 .groupby(["dataset_source", "Site", "PlotID"])
                 .agg(n_arboles=("treeID", "size"),
                      n_arboles_con_censo3=("_c3", "sum"),
                      fecha_censo3=("date_census3", "max"))
                 .reset_index())

resumen_plots["recensado"] = resumen_plots["n_arboles_con_censo3"] > 0
resumen_plots["forzado"]   = resumen_plots["PlotID"].isin(PLOTS_FORZADOS)
resumen_plots["se_conserva"] = resumen_plots["recensado"] | resumen_plots["forzado"]

print(resumen_plots.to_string(index=False))

faltantes = [p for p in PLOTS_FORZADOS if p not in set(resumen_plots["PlotID"])]
if faltantes:
    print(f"\n[!] OJO: estas parcelas de PLOTS_FORZADOS no existen en las bases: {faltantes}")

     dataset_source       Site PlotID  n_arboles  n_arboles_con_censo3 fecha_censo3  recensado  forzado  se_conserva
            Galeras    Galeras GAL_20         31                     0          NaT      False     True         True
            Galeras    Galeras GAL_21         19                     0          NaT      False    False        False
            Galeras    Galeras GAL_22         45                    28   2024-09-06       True    False         True
            Galeras    Galeras GAL_23         39                    31   2024-09-06       True    False         True
            Galeras    Galeras GAL_24         36                    24   2025-01-08       True    False         True
            Galeras    Galeras GAL_25         37                     0          NaT      False    False        False
            Galeras    Galeras GAL_26         33                     0          NaT      False    False        False
            Galeras    Galeras GAL_28         60                

In [18]:
plots_conservados = resumen_plots.loc[resumen_plots["se_conserva"], "PlotID"]

n_antes = df_todo["PlotID"].nunique()
df_unificado = (df_todo[df_todo["PlotID"].isin(plots_conservados)]
                .reset_index(drop=True))
df_unificado["recenso_forzado"] = df_unificado["PlotID"].isin(PLOTS_FORZADOS)

if not INCLUIR_COLS_CALIDAD:
    df_unificado = df_unificado.drop(columns=[f"date_census{n}_origen" for n in (1, 2, 3)])

n_despues = df_unificado["PlotID"].nunique()
eliminados = sorted(set(resumen_plots.loc[~resumen_plots["se_conserva"], "PlotID"]))

print(f"Plots totales: {n_antes}  ->  Plots conservados: {n_despues}")
print(f"Filas totales: {df_todo.shape[0]}  ->  Filas conservadas: {df_unificado.shape[0]}")
print(f"\nPlots eliminados (sin recenso): {', '.join(eliminados) if eliminados else 'ninguno'}")
print(f"Plots conservados sin recenso (forzados): "
      f"{', '.join(sorted(set(df_unificado.loc[df_unificado['recenso_forzado'], 'PlotID'])))}")
print("\nAños del censo 3 en la base final:")
print(df_unificado["year_census3"].value_counts(dropna=False).sort_index().to_string())

Plots totales: 51  ->  Plots conservados: 30
Filas totales: 2113  ->  Filas conservadas: 1332

Plots eliminados (sin recenso): GAL_21, GAL_25, GAL_26, GAL_47, GAL_49, GAL_61, GCM_27, GCM_52, GCM_53, GCM_54, SEV_71, SEV_73, SEV_75, SUM_14, SUM_15, SUM_17, SUM_19, SUM_56, SUM_57, SUM_58, SUM_59
Plots conservados sin recenso (forzados): GAL_20, GAL_29, SUM_13

Años del censo 3 en la base final:
year_census3
2023    355
2024    302
2025    357
<NA>    318


## 11. Control de calidad: resumen por fuente

In [19]:
summary = df_unificado.groupby("dataset_source").agg(
    n_filas=("treeID", "size"),
    n_plots=("PlotID", "nunique"),
    n_familias=("family", "nunique"),
    con_dbh_census1=("dbh_census1", lambda s: s.notna().sum()),
    con_dbh_census2=("dbh_census2", lambda s: s.notna().sum()),
    con_dbh_census3=("dbh_census3", lambda s: s.notna().sum()),
    muertos_census3=("dead_census3", lambda s: (s == 1).sum()),
).reset_index()
summary

,dataset_source,n_filas,n_plots,n_familias,con_dbh_census1,con_dbh_census2,con_dbh_census3,muertos_census3
0,Galeras,456,10,52,368,66,285,14
1,Guacamayos_Oyacachi,355,8,34,233,188,226,126
2,SelvaViva,246,5,44,192,123,172,6
3,Sumaco,275,7,30,157,105,116,4


In [20]:
# porcentaje de valores faltantes por columna, para revision rapida
missing_pct = (df_unificado.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct.to_frame("pct_faltante")

,pct_faltante
tree_notes,99.4
comment_census2,97.5
censos_intermedios,90.1
comment_census1,85.1
dead_census1,82.6
Subplot,77.9
Old_Tag,75.7
dead_census3,71.5
comment_taxo,70.2
dbh_census2,63.8


In [21]:
# rango de DBH por base: sirve para detectar problemas de unidades (mm vs cm)
print(df_unificado.groupby("dataset_source")[COLS_DBH]
      .agg(["min", "median", "max"]).round(1).to_string())

                    dbh_census1               dbh_census2               dbh_census3              
                            min median    max         min median    max         min median    max
dataset_source                                                                                   
Galeras                     9.9   16.8  129.9        10.0   18.8  112.7         9.0   17.6  168.4
Guacamayos_Oyacachi        10.0   18.5   66.8        10.0   18.0   56.4         2.3   19.0   80.1
SelvaViva                  10.1   16.9   85.7        10.0   14.9   86.6        10.0   18.1   88.8
Sumaco                      9.9   19.2  196.1        10.2   20.7  196.7        10.0   20.6  187.8


## 12. Control: columnas de rasgos descartadas

Se listan las columnas de cada archivo original que **no** pasaron al esquema comun.
Deberian ser todas de rasgos / muestras / herbario / alturas, que ya estan en la base de
rasgos. Si aparece aqui alguna columna de dinamica forestal que si haga falta, hay que
agregarla a la funcion `standardize_*` correspondiente.

In [22]:
for name, df_orig in raw.items():
    descartadas = columnas_descartadas(df_orig, COLUMNAS_USADAS[name])
    print(f"{name} ({len(descartadas)} columnas descartadas):")
    for c in descartadas:
        print(f"    - {c}")
    print()

print("Guacamayos_Oyacachi: de las hojas ForestPlots se descartaron los campos")
print("    - voucher code / voucher collected / leaf sample tree  (muestras y herbario)")
print("    - Height / Height Broken At / Flag5                    (altura)")
print("    - POM, Flag3, Flag4, T1, T2, X, Y                      (posicion y detalle de medicion)")
print("    - Individual tree census 1 / 2                         (control interno de ForestPlots)")

Galeras (21 columnas descartadas):
    - Individual tree census 1
    - Individual tree census 2
    - new census 2
    - tree height 2011 (m)
    - JH herbarium collections
    - leaf sample tree
    - Select_2024
    - Collected_2024
    - ind (1) branch (0)
    - dead 2011
    - height (m) 2024
    - leaf sample tree 3
    - new entry 2024
    - X/Y
    - Cambio sp
    - JH herbarium collections 2024
    - Nº duplicados/ muestras 2025
    - crust thickness
    - wet lenght cm
    - wet wood weight
    - dry wood weight

SelvaViva (14 columnas descartadas):
    - Individual tree census 1
    - new census 2
    - dendrometer
    - tree height 2011 (m)
    - JH herbarium collections
    - leaf sample 2025
    - wood sample 2025
    - ind (1) branch (0)
    - dead 2011
    - height (m) 2025
    - new entry 2025
    - Cambio sp
    - JH herbarium collections 2025
    - Nº duplicados/ muestras 2025

Sumaco (15 columnas descartadas):
    - Individual tree census 1
    - Individual tree cen

## 13. Guardar la base unificada

Esta base queda lista para **unirse despues con la base de rasgos** ya trabajada,
usando como llaves de union `PlotID` + `treeID` (o `new_tree_ID` cuando el `treeID`
original no exista).

In [23]:
out_xlsx = OUT_DIR / "DinamicaForestal_unificada_recensos.xlsx"
out_csv  = OUT_DIR / "DinamicaForestal_unificada_recensos.csv"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as xw:
    df_unificado.to_excel(xw, sheet_name="Dinamica_unificada", index=False)
    resumen_plots.to_excel(xw, sheet_name="Resumen_plots", index=False)
    reporte_fechas.to_excel(xw, sheet_name="Reporte_fechas", index=False)
df_unificado.to_csv(out_csv, index=False)

print(f"Guardado: {out_xlsx}")
print(f"Guardado: {out_csv}")
print(f"Guardado: {OUT_GCM_OYA}  (base intermedia Guacamayos + Oyacachi)")
print(f"\nBase final: {df_unificado.shape[0]} filas x {df_unificado.shape[1]} columnas")

Guardado: DinamicaForestal_unificada_recensos.xlsx
Guardado: DinamicaForestal_unificada_recensos.csv
Guardado: Base_unificada_ForestPlots_Guacamayos_Oyacachi.xlsx  (base intermedia Guacamayos + Oyacachi)

Base final: 1332 filas x 34 columnas
